# A.X-3.1-Light QLoRA 파인튜닝

**목적**: 베이스라인 실험에서 발견된 약점(복수 엔티티 혼동, 열거 조기 종료, 세부 정보 누락) 보강  
**환경**: Google Colab L4 GPU (24GB VRAM)  
**데이터**: `ai/data/ft_train_data.json` (500건) → train 450 / val 50  
**평가**: `ai/data/qa_samples.json` (40건, 베이스라인과 동일 조건)

| 설정 | 값 |
|------|----|
| 베이스 모델 | skt/A.X-3.1-Light (7B) |
| 양자화 | QLoRA NF4 4-bit |
| LoRA rank | 16 |
| LoRA alpha | 32 |
| Learning rate | 2e-4 |
| Epochs | 3 |
| Effective batch size | 16 (4 × 4 accumulation) |

In [ ]:
# 셀 1: 저장소 클론 및 의존성 설치
!git clone https://github.com/SKNETWORKS-FAMILY-AICAMP/SKN21-FINAL-3TEAM.git
%cd SKN21-FINAL-3TEAM

!pip install -q transformers peft trl bitsandbytes accelerate rouge-score datasets

In [ ]:
# 셀 2: 이미 클론한 경우 최신 코드 pull
# (위 셀 1을 실행했으면 이 셀은 건너뛰세요)
import os
if os.path.exists('SKN21-FINAL-3TEAM'):
    %cd SKN21-FINAL-3TEAM
    !git pull

In [ ]:
# 셀 3: HuggingFace 인증 (Colab Secrets에서 HF_TOKEN 읽기)
import os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('HF_TOKEN 설정 완료')

In [ ]:
# 셀 4: PYTHONPATH 설정
import sys, os
sys.path.insert(0, os.getcwd())
print('PYTHONPATH:', os.getcwd())

In [ ]:
# 셀 5: GPU 및 데이터 확인
import torch, json

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

with open('ai/data/ft_train_data.json', encoding='utf-8') as f:
    train_data = json.load(f)
print(f'\n학습 데이터: {len(train_data)}건')
for t in ['multi_entity', 'enumeration', 'detail_missing', 'general_qa']:
    n = sum(1 for s in train_data if s.get('type') == t)
    print(f'  {t}: {n}건')

with open('ai/data/qa_samples.json', encoding='utf-8') as f:
    test_data = json.load(f)
print(f'\n평가 데이터: {len(test_data)}건 (베이스라인과 동일)')

In [ ]:
# 셀 6: 학습 실행
# ※ A.X-3.1-Light 7B, QLoRA 4-bit → 약 10~12GB VRAM 사용
# ※ 500건 x 3 epoch → L4 기준 약 30~40분 소요
!python ai/finetuning/train_qa_lora.py --mode train

In [ ]:
# 셀 7: 평가 실행 (베이스라인 qa_samples.json 40건 기준)
!python ai/finetuning/train_qa_lora.py --mode eval --adapter_path ai/finetuning/output/final

In [ ]:
# 셀 8: 평가 결과 상세 출력
import json

with open('ai/finetuning/output/ft_eval_results.json', encoding='utf-8') as f:
    results = json.load(f)

# 베이스라인 A.X 약점 케이스 집중 확인
weak_ids = ['biz_007', 'gen_018', 'biz_013']
print('=== 베이스라인 약점 케이스 ===')    
for r in results:
    if r['id'] in weak_ids:
        print(f"\n[{r['id']}]")
        print(f"  Q   : {r['question']}")
        print(f"  Gold: {r['gold_answer']}")
        print(f"  Pred: {r['pred_answer']}")
        print(f"  TF1={r['token_f1']:.3f}  RL={r['rouge_l']:.3f}")

In [ ]:
# 셀 9: 결과 파일 다운로드
from google.colab import files
files.download('ai/finetuning/output/ft_eval_results.json')

# 어댑터도 압축 후 다운로드
!zip -r ft_adapter.zip ai/finetuning/output/final/
files.download('ft_adapter.zip')